# Code Generator

The requirement: use a Frontier model to generate a Javascript or Python code from your idea


In [ ]:
# imports

import os
import io
import re
import sys
import contextlib
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [ ]:
load_dotenv(override=True)
# API Keys (Only free/open-source ones needed)
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# Client Setup
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"



In [ ]:
# Connect to client libraries

groq = OpenAI(api_key=groq_api_key, base_url=groq_url) if groq_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url) if openrouter_api_key else None



In [ ]:
# Free / Open-Source Models Configuration
models = [
    # Working Groq Free Tier Endpoints
    "openai/gpt-oss-120b (Groq)",
    "openai/gpt-oss-20b (Groq)",
    
    # Active OpenRouter Free Models (:free suffix required)
    "qwen/qwen3.8-27b:free (OpenRouter)",
    "poolside/laguna-s-2.1:free (OpenRouter)",
    "cohere/north-mini-code:free (OpenRouter)",
    "google/gemma-4-31b-it:free (OpenRouter)",
    "openrouter/free (OpenRouter)"  # Dynamic router fallback
]

clients = {
    # Groq Client Definitions
    "openai/gpt-oss-120b (Groq)": (groq, "openai/gpt-oss-120b"),
    "openai/gpt-oss-20b (Groq)": (groq, "openai/gpt-oss-20b"),
    
    # OpenRouter Client Definitions
    "qwen/qwen3.8-27b:free (OpenRouter)": (openrouter, "qwen/qwen3.8-27b:free"),
    "poolside/laguna-s-2.1:free (OpenRouter)": (openrouter, "poolside/laguna-s-2.1:free"),
    "cohere/north-mini-code:free (OpenRouter)": (openrouter, "cohere/north-mini-code:free"),
    "google/gemma-4-31b-it:free (OpenRouter)": (openrouter, "google/gemma-4-31b-it:free"),
    "openrouter/free (OpenRouter)": (openrouter, "openrouter/free"),
}

In [ ]:
def system_prompt_for(language):
    return f""" You are an expert software engineer.
Your task is to implement the requested idea in clean, runnable {language} code.
Respond ONLY with raw executable code. Do not include markdown code blocks (``` or ```{language.lower()}), extra formatting, or conversational text.
Do NOT use terminal color/formatting packages (such as chalk, colors, colorama) or raw ANSI escape codes. Output plain text only.
Include a simple driver script at the end that prints output so the user can verify it works.
"""

In [ ]:
def generate_code(model_name, user_idea, language):
    client, real_model_id = clients[model_name]
    
    if not client:
        return f"Error: API key or client for {model_name} is not configured."

    messages = [
        {"role": "system", "content": system_prompt_for(language)},
        {"role": "user", "content": f"Implement the following feature/idea in {language}:\n{user_idea}"}
    ]

    try:
        response = client.chat.completions.create(
            model=real_model_id,
            messages=messages
        )
        reply = response.choices[0].message.content or ""

        # Robustly strip top/bottom backtick wrappers
        lines = reply.strip().splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]

        return "\n".join(lines).strip()
    except Exception as e:
        return f"Error generating code: {e}"

In [ ]:


# Regex pattern to strip ANSI escape codes
ANSI_ESCAPE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')

def strip_ansi(text):
    return ANSI_ESCAPE.sub('', text) if text else text

def execute_code(code, language):
    if not code or not code.strip():
        return "No code to run."

    # --- PYTHON EXECUTION ---
    if language == "Python":
        buffer = io.StringIO()
        try:
            with contextlib.redirect_stdout(buffer):
                # Explicitly set __name__ to '__main__' so top-level script blocks execute
                exec_globals = {
                    "__builtins__": __builtins__,
                    "__name__": "__main__"
                }
                exec(code, exec_globals)
            output = buffer.getvalue()
        except Exception as e:
            output = f"Python Execution Error:\n{e}"

        clean_out = strip_ansi(output)
        return clean_out if clean_out.strip() else "Code executed successfully with no stdout output."

    # --- JAVASCRIPT EXECUTION ---
    elif language == "JavaScript":
        js_file = "generated_app.js"
        with open(js_file, "w", encoding="utf-8") as f:
            f.write(code)

        # Force Node.js to disable ANSI color formatting
        env = os.environ.copy()
        env["FORCE_COLOR"] = "0"
        env["NODE_DISABLE_COLORS"] = "1"

        try:
            run_result = subprocess.run(
                ["node", js_file], 
                check=True, 
                text=True, 
                capture_output=True,
                env=env
            )
            output = run_result.stdout if run_result.stdout.strip() else "Code executed successfully with no stdout output."
            return strip_ansi(output)
        except subprocess.CalledProcessError as e:
            return f"JavaScript (Node.js) Error:\n{strip_ansi(e.stderr)}"
        except FileNotFoundError:
            return "Error: Node.js is not installed or not added to your system PATH."

In [ ]:
# UI Layout
default_idea = "Create a function to calculate the total price of items in a shopping cart with discounts applied, then print the result."

try:
    from styles import CSS
except ImportError:
    CSS = ""

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Idea-to-Code Generator") as ui:
    gr.Markdown("# Idea to Code Generator")
    
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            idea_input = gr.TextArea(
                label="Describe your idea or feature functionality",
                value=default_idea,
                lines=12
            )
            target_language = gr.Radio(
                choices=["Python", "JavaScript"],
                value="Python",
                label="Target Programming Language"
            )
        with gr.Column(scale=6):
            generated_code = gr.Code(
                label="Generated Code",
                value="",
                language="python",
                lines=16
            )

    with gr.Row(elem_classes=["controls"]):
        model_dropdown = gr.Dropdown(
            choices=models, 
            value=models[0], 
            show_label=False
        )
        generate_btn = gr.Button("Generate Code", elem_classes=["convert-btn"])
        run_btn = gr.Button("Run Generated Code", elem_classes=["run-btn"])

    with gr.Row(equal_height=True):
        output_terminal = gr.TextArea(
            label="Execution Result Output", 
            lines=8, 
            elem_classes=["py-out"]
        )

    # Dynamic Syntax Highlighting
    def update_code_language(lang):
        lang_map = {"Python": "python", "JavaScript": "javascript"}
        return gr.Code(language=lang_map.get(lang, "python"))

    target_language.change(
        fn=update_code_language, 
        inputs=[target_language], 
        outputs=[generated_code]
    )

    # Event Bindings
    generate_btn.click(
        fn=generate_code, 
        inputs=[model_dropdown, idea_input, target_language], 
        outputs=[generated_code]
    )
    
    run_btn.click(
        fn=execute_code, 
        inputs=[generated_code, target_language], 
        outputs=[output_terminal]
    )

ui.launch(inbrowser=True)